In [4]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import pandas as pd
import numpy as np

# Transforming World Development Indicators (WDI)

## current version

In [45]:
# loading up the data
WDI_full = pd.read_excel(r'data\raw\WDI_11_07_2025.xlsx')

In [ ]:
# loading up the list of desired data
WDI_meta = ['CM.MKT.LCAP.CD'
            , 'CM.MKT.LCAP.GD.ZS'
            , 'CM.MKT.TRAD.CD'
            , 'CM.MKT.TRAD.GD.ZS'
            , 'FB.BNK.CAPA.ZS'
            , 'NY.GDP.MKTP.KD'
            , 'SP.POP.TOTL'
            , 'FS.AST.PRVT.GD.ZS'
            , 'NY.GDP.MKTP.PP.KD'
            ]

In [46]:
# loading up the list of desired data
WDI_meta = pd.read_excel(r'data\meta\WDI_meta.xlsx')['Series Code']

In [47]:
# taking the necescary data and turning into tablular data
WDI_full = WDI_full[WDI_full['Indicator Code'].isin(WDI_meta)]
WDI_full = pd.melt(WDI_full, id_vars=list(WDI_full.columns)[:4], value_vars=list(WDI_full.columns)[4:])

In [48]:
# renaming
WDI_full.rename(columns={'variable': 'Year'
                          , 'value': 'Value'
                         }
                 , inplace=True)

In [49]:
# making sure the data is an number
WDI_full['Value'] = pd.to_numeric(WDI_full['Value'], errors='coerce')

In [50]:
# taking only the necescary years
WDI_full['Year'] = WDI_full['Year'].astype(int)
WDI_full = WDI_full[WDI_full['Year']>1989]

In [51]:
WDI_full['Source']= 'World Bank 2025'

In [52]:
WDI_full = WDI_full.dropna(subset=['Value'])

## country name formatting

In [60]:
# making sure the names are propper
WDI_full['Country Name'].replace('Turkiye','Turkey',inplace=True)
WDI_full['Country Name'].replace('Egypt', 'Egypt, Arab Rep.',inplace=True)
WDI_full['Country Name'].replace('Russia','Russian Federation',inplace=True)
WDI_full['Country Name'].replace('''Côte d'Ivoire''','''Cote d'Ivoire''',inplace=True)
WDI_full['Country Name'].replace('Czechia','Czech Republic',inplace=True)
WDI_full['Country Name'].replace('Iran','Iran, Islamic Rep.',inplace=True)
WDI_full['Country Name'].replace('Korea','Korea, Rep.',inplace=True)
WDI_full['Country Name'].replace('Eswatini','Swaziland',inplace=True)
WDI_full['Country Name'].replace('Viet Nam','Vietnam',inplace=True)

In [61]:
WDI_full = WDI_full.drop_duplicates(subset=WDI_full.columns[:-2], keep='last')

# Combining the raw dfs

In [ ]:
# loading up data from a previous study
WDI_GFDI_past_study = pd.read_excel(r'data\hand filled\WDI_GFDI_past_study.xlsx')

In [ ]:
# loading up data from current study
WDI_GFDI_current_study = pd.read_excel(r'data\hand filled\WDI_GFDI_current_study.xlsx')

In [ ]:
# everything together
WDI_full_everything = pd.concat([WDI_GFDI_past_study, WDI_GFDI_current_study, WDI_full])

In [ ]:
# making sure the names are propper
WDI_full_everything['Country Name'].replace('Turkiye','Turkey',inplace=True)
WDI_full_everything['Country Name'].replace('Egypt', 'Egypt, Arab Rep.',inplace=True)
WDI_full_everything['Country Name'].replace('Russia','Russian Federation',inplace=True)
WDI_full_everything['Country Name'].replace('''Côte d'Ivoire''','''Cote d'Ivoire''',inplace=True)
WDI_full_everything['Country Name'].replace('Czechia','Czech Republic',inplace=True)
WDI_full_everything['Country Name'].replace('Iran','Iran, Islamic Rep.',inplace=True)
WDI_full_everything['Country Name'].replace('Korea','Korea, Rep.',inplace=True)
WDI_full_everything['Country Name'].replace('Eswatini','Swaziland',inplace=True)
WDI_full_everything['Country Name'].replace('Viet Nam','Vietnam',inplace=True)

In [ ]:
WDI_full_everything = WDI_full_everything.dropna(subset=['Value'])

In [ ]:
WDI_full_everything = WDI_full_everything[['Country Name'
                                           , 'Country Code'
                                           , 'Indicator Name'
                                           , 'Indicator Code'
                                           , 'Year'
                                           , 'Value'
                                           , 'Source'
                                          ]]

In [ ]:
WDI_full_everything_WB = WDI_full_everything[WDI_full_everything['Source'].str.startswith('World Bank')].drop_duplicates(subset=['Country Name'
                                                                                                                    , 'Country Code'
                                                                                                                    , 'Indicator Name'
                                                                                                                    , 'Indicator Code'
                                                                                                                    , 'Year']
                                                                                                            , keep = 'last')

In [ ]:
WDI_full_non_WB = WDI_full_everything[~WDI_full_everything['Source'].str.startswith('World Bank')].drop_duplicates(subset=['Country Name'
                                                                                                                    , 'Country Code'
                                                                                                                    , 'Indicator Name'
                                                                                                                    , 'Indicator Code'
                                                                                                                    , 'Year']
                                                                                                            , keep = 'last')

In [ ]:
WDI_full = pd.concat([WDI_full_non_WB, WDI_full_everything_WB]).drop_duplicates(subset=['Country Name'
                                                                                        , 'Country Code'
                                                                                        , 'Indicator Name'
                                                                                        , 'Indicator Code'
                                                                                        , 'Year']
                                                                                , keep = 'last')

In [ ]:
WDI_full

In [72]:
# a piovt table to see how many entries of each variable
wdi_piovt = WDI_full.pivot_table(index=['Country Code'],
                                 columns=['Indicator Name'],
                                 values=['Value'],
                                 aggfunc='count')#.to_excel(r'wdi_piovt.xlsx')

In [73]:
# if a country has less than 8 entries set as none and drop those
wdi_piovt = wdi_piovt.applymap(lambda x: None if x < 8 else x).dropna()

In [74]:
# and thus we get a list of countries which have at least 8 entries of each variable
country_codes = list(wdi_piovt.index.get_level_values(0))

In [76]:
len(country_codes)

68

In [77]:
WDI_full_target = WDI_full[WDI_full['Country Code'].isin(country_codes)]

In [78]:
# bringing in extra info about the countries (their region and income groups
WDI_full_target = pd.merge(WDI_full_target, GFDI_country, how='left', on='Country Code')

In [79]:
# reoredring the columns
WDI_full_target = WDI_full_target[['Country Name'
                                   , 'Country Code'
                                   , 'region'
                                   , 'income'
                                   , 'Indicator Name'
                                   , 'Indicator Code'
                                   , 'Year'
                                   , 'Value'


                                   ,'Source'
                                  ]]

# adding the hand collected data to the filtered raw df

In [80]:
target_columns = ['Bank deposits to GDP (%)'
                  , 'Domestic credit to private sector (% of GDP)'
                  , 'Broad money (% of GDP)'
                  , 'Stocks traded, total value (current US$)'
                  , 'Market capitalization of listed domestic companies (current US$)'
                  , 'GDP (current US$)'
                  , 'GDP per capita, PPP (constant 2021 international $)'
                  , 'GDP, PPP (constant 2021 international $)'
                 ]

In [81]:
WDI_full_target = WDI_full_target[WDI_full_target['Indicator Name'].isin(target_columns)]

In [82]:
# loading up data from a previous study
WDI_full_target.to_excel(r'data\WDI_full_collected.xlsx', index=False)